# Introdução

Esse notebook tem o objetivo de analisar o desempenho dos modelos forecasting, pesquisar e tentar entender padrões mais profundos dos dados.

# Código

In [1]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import glob
from itertools import product
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
app_path = os.path.abspath(os.path.join('..', 'app'))
if app_path not in sys.path:
    sys.path.append(app_path)

from app.database import SessionLocal
from sqlalchemy import text

In [2]:
def carregar_relatorios():
    arquivos = sorted(glob.glob("data/relatorio_*.csv"))
    dfs = []

    for arquivo in arquivos:
        nome_arquivo = os.path.basename(arquivo)
        data_str = (
            nome_arquivo
            .replace("relatorio_", "")
            .replace(".csv", "")
        )
        data_arquivo = pd.to_datetime(
            data_str,
            format="%d-%m-%Y"
        ).strftime("%Y-%m-%d")
        df = pd.read_csv(arquivo)
        df = df.reset_index()
        df = df.drop(columns=("Área1-Pessoas que entraram"))
        df.columns = ['timestamp', 'fluxo']
        df['hora_inicio'] = df['timestamp'].str.split('-').str[0]
        df['timestamp'] = pd.to_datetime(
            data_arquivo + ' ' + df['hora_inicio']
        )
        df.set_index('timestamp', inplace=True)
        df.drop(columns=['hora_inicio'], inplace=True)
        dfs.append(df)
    df_final = pd.concat(dfs)
    df_final = df_final.sort_index()
    df_final.index.freq = 'h'

    return df_final

db = SessionLocal()

map_lojas = {
    205709335: "Vans",
    205709338: "Arezzo",
    205785185: "Off Premium",
    206057004: "Ida",
    205613392: "Aramis",
    205406209: "High",
    206057013: "Vix",
    206057007: "Surto dos 50"
}

try:
    query = text("""
        SELECT id_loja, quantidade, timestamp
        FROM vendas_itens
        WHERE timestamp BETWEEN '2026-05-07 00:00:00'
        AND '2026-05-08 23:59:59'
    """)

    df = pd.read_sql_query(query, db.bind)

    df["loja"] = df["id_loja"].map(map_lojas)

    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.floor("h")

    # Agrupa
    df = (
        df.groupby(["loja", "timestamp"])["quantidade"]
        .sum()
        .reset_index()
    )

    # Range completo de horas
    full_range = pd.date_range(
        start="2026-05-07 00:00:00",
        end="2026-05-08 23:00:00",
        freq="h"
    )

    # Cria todas combinações loja x hora
    idx = pd.MultiIndex.from_product(
        [df["loja"].unique(), full_range],
        names=["loja", "timestamp"]
    )

    # Reindex correto
    df = (
        df.set_index(["loja", "timestamp"])
        .reindex(idx, fill_value=0)
        .reset_index()
    )

    df.rename(columns={"quantidade": "fluxo"}, inplace=True)

    df_vendas_real = df.sort_values(["loja", "timestamp"])

    print("Dados carregados com sucesso!")
    print(df_vendas_real.head())

except Exception as e:
    print(f"Erro ao carregar os dados: {e}")

finally:
    db.close()

df_fluxo_real = carregar_relatorios()
print(df_fluxo_real.head())

Dados carregados com sucesso!
     loja           timestamp  fluxo
0  Aramis 2026-05-07 00:00:00    0.0
1  Aramis 2026-05-07 01:00:00    0.0
2  Aramis 2026-05-07 02:00:00    0.0
3  Aramis 2026-05-07 03:00:00    0.0
4  Aramis 2026-05-07 04:00:00    0.0
                     fluxo
timestamp                 
2026-05-06 00:00:00      5
2026-05-06 01:00:00      0
2026-05-06 02:00:00      0
2026-05-06 03:00:00      0
2026-05-06 04:00:00      0


In [3]:
db = SessionLocal()

try:
    query = text("""
        SELECT loja, previsao, timestamp_previsao
        FROM previsoes_vendas
        WHERE timestamp_previsao BETWEEN '2026-05-07 00:00:00'
        AND '2026-05-08 23:59:59'
    """)

    df = pd.read_sql_query(query, db.bind)

    df["timestamp_previsao"] = pd.to_datetime(
        df["timestamp_previsao"]
    ).dt.floor("h")

    full_range = pd.date_range(
        start="2026-05-06 00:00:00",
        end="2026-05-07 23:00:00",
        freq="h"
    )

    lojas = df["loja"].unique()

    full_index = pd.MultiIndex.from_product(
        [lojas, full_range],
        names=["loja", "timestamp_previsao"]
    )

    df = (
        df.set_index(["loja", "timestamp_previsao"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    df_vendas_previsao = df[df["loja"] != "Fluxo"]
    df_fluxo_previsao = df[df["loja"] == "Fluxo"]

    print("Dados carregados com sucesso!")

except Exception as e:
    print(f"Erro ao carregar os dados: {e}")

finally:
    db.close()


print(df_vendas_previsao.head())
print(df_fluxo_previsao.head())

Dados carregados com sucesso!
     loja  timestamp_previsao  previsao
0  Aramis 2026-05-06 00:00:00       0.0
1  Aramis 2026-05-06 01:00:00       0.0
2  Aramis 2026-05-06 02:00:00       0.0
3  Aramis 2026-05-06 03:00:00       0.0
4  Aramis 2026-05-06 04:00:00       0.0
      loja  timestamp_previsao  previsao
384  Fluxo 2026-05-06 00:00:00       0.0
385  Fluxo 2026-05-06 01:00:00       0.0
386  Fluxo 2026-05-06 02:00:00       0.0
387  Fluxo 2026-05-06 03:00:00       0.0
388  Fluxo 2026-05-06 04:00:00       0.0


In [4]:
df_fluxo_comparacao = df_fluxo_real.merge(
    df_fluxo_previsao,
    left_index=True,
    right_on="timestamp_previsao",
    how="inner"
)

df_fluxo_comparacao = df_fluxo_comparacao[(df_fluxo_comparacao['timestamp_previsao'] >= '2026-05-07 10:00:00') &
                                            (df_fluxo_comparacao['timestamp_previsao'] <= '2026-05-07 23:00:00')]

mae = np.mean(
    np.abs(df_fluxo_comparacao['fluxo'] - df_fluxo_comparacao['previsao'])
)

wape = (
    np.abs(df_fluxo_comparacao['fluxo'] - df_fluxo_comparacao['previsao']).sum()
    / df_fluxo_comparacao['fluxo'].abs().sum()
) * 100


print(f"MAE: {mae}")
print(f"WAPE: {wape:.2f}%")

MAE: 446.5938757302054
WAPE: 87.26%


In [5]:
df_vendas_comparacao = df_vendas_real.merge(
    df_vendas_previsao,
    left_on=['loja', 'timestamp'],
    right_on=['loja', 'timestamp_previsao'],
    how='inner'
)

df_vendas_comparacao = df_vendas_comparacao[(df_vendas_comparacao['timestamp'] >= '2026-05-07 10:00:00') & (df_vendas_comparacao['timestamp'] <= '2026-05-08 23:00:00')]

mae_agrupado_vendas = df_vendas_comparacao.groupby('loja').apply(
    lambda x: np.mean(np.abs(x['fluxo'] - x['previsao']))
)

wape_agrupado_vendas = df_vendas_comparacao.groupby('loja').apply(
    lambda x: (np.abs(x['fluxo'] - x['previsao']).sum() / x['fluxo'].abs().sum()) * 100
)

mae_vendas = np.mean(
    np.abs(df_vendas_comparacao['fluxo'] - df_vendas_comparacao['previsao'])
)

wape_vendas = (
    np.abs(df_vendas_comparacao['fluxo'] - df_vendas_comparacao['previsao']).sum()
    / df_vendas_comparacao['fluxo'].abs().sum()
) * 100

print(f"MAE Vendas: {mae_vendas}")
print(f"WAPE Vendas: {wape_vendas:.2f}%")
print(f"MAE Agrupado Vendas:\n{mae_agrupado_vendas}")
print(f"WAPE Agrupado Vendas:\n{wape_agrupado_vendas}")

MAE Vendas: 16.962971444880463
WAPE Vendas: 142.42%
MAE Agrupado Vendas:
loja
Aramis          24.642705
Arezzo           8.988727
High            30.696128
Ida             22.341904
Off Premium      7.928572
Surto dos 50    30.517610
Vans             9.250001
Vix              1.338125
dtype: float64
WAPE Agrupado Vendas:
loja
Aramis           86.465631
Arezzo           93.912076
High            228.588185
Ida              75.552333
Off Premium     124.719106
Surto dos 50    928.796819
Vans            264.285754
Vix             124.891641
dtype: float64


In [15]:
# Comparação vendas antes de 18h e depois de 18h por loja

df_vendas_real['periodo'] = df_vendas_real['timestamp'].dt.hour.apply(
    lambda x: 'Antes de 18h' if x < 18 else 'Depois de 18h'
)

tabela_comparacao = df_vendas_real.groupby(['loja', 'periodo'])['fluxo'].sum().unstack().fillna(0)

tabela_comparacao['Total'] = tabela_comparacao.sum(axis=1)
tabela_comparacao['% Depois de 18h'] = (tabela_comparacao['Depois de 18h'] / tabela_comparacao['Total']) * 100

map_lojas_ticket_medio = {
    "Vans": 200,
    "Arezzo": 180,
    "Off Premium": 320,
    "Ida": 265,
    "Aramis": 452,
    "High": 230,
    "Vix": 480.0,
    "Surto dos 50": 107
}

tabela_comparacao['Ticket Médio'] = tabela_comparacao.index.map(map_lojas_ticket_medio)
tabela_comparacao['Faturamento Antes de 18h'] = tabela_comparacao['Antes de 18h'] * tabela_comparacao['Ticket Médio']
tabela_comparacao['Faturamento Depois de 18h'] = tabela_comparacao['Depois de 18h'] * tabela_comparacao['Ticket Médio']
tabela_comparacao = tabela_comparacao.sort_values('% Depois de 18h', ascending=False)
print(tabela_comparacao[['Antes de 18h', 'Depois de 18h', '% Depois de 18h', 'Faturamento Antes de 18h', 'Faturamento Depois de 18h']])
media_percentual_depois_18h = tabela_comparacao['% Depois de 18h'].mean()
media_percentual_depois_18h_faturamento = (tabela_comparacao['Faturamento Depois de 18h'].sum() / (tabela_comparacao['Faturamento Antes de 18h'].sum() + tabela_comparacao['Faturamento Depois de 18h'].sum())) * 100
print(f"Média percentual de vendas depois de 18h: {media_percentual_depois_18h:.2f}%")
print(f"Média percentual de faturamento depois de 18h: {media_percentual_depois_18h_faturamento:.2f}%")

periodo       Antes de 18h  Depois de 18h  % Depois de 18h  \
loja                                                         
Aramis               195.0          368.0        65.364121   
High                 108.0          158.0        59.398496   
Off Premium           80.0           98.0        55.056180   
Vans                  59.0           56.0        48.695652   
Surto dos 50          53.0           48.0        47.524752   
Ida                  455.0          343.0        42.982456   
Arezzo               153.0           92.0        37.551020   
Vix                   21.0            4.0        16.000000   

periodo       Faturamento Antes de 18h  Faturamento Depois de 18h  
loja                                                               
Aramis                         88140.0                   166336.0  
High                           24840.0                    36340.0  
Off Premium                    25600.0                    31360.0  
Vans                           11800.0 